In [0]:
# COMMAND ----------
# STEP 1: INITIALIZE LIBRARIES & DEFINE ENVIRONMENTAL CONFIGURATIONS
# COMMAND ----------
from pyspark.sql.functions import col, current_timestamp

# 1. Define Databricks Widgets (Sets up defaults for manual interactive runs)
dbutils.widgets.text("env_catalog", "dev_catalog", "1. Catalog Name")
dbutils.widgets.text("env_schema", "customer_analytics", "2. Schema Name")
dbutils.widgets.text("base_volume_path", "/Volumes", "3. Base Volume Root Path")

# 2. Extract values dynamically
catalog = dbutils.widgets.get("env_catalog")
schema = dbutils.widgets.get("env_schema")
base_path = dbutils.widgets.get("base_volume_path")

# 3. Construct unified targets
VOLUME_PATH = f"{base_path}/{catalog}/{schema}/landing"
TARGET_SCHEMA = f"{catalog}.{schema}"

print(f"Target Schema: {TARGET_SCHEMA}")
print(f"Volume Landing Path: {VOLUME_PATH}")

In [0]:
def ingest_and_archive_entity(entity_name: str):
    """
    Ingests Parquet files from a specific landing folder into a Bronze Delta table,
    adds metadata, and archives processed files.
    """
    landing_path = f"{VOLUME_PATH}/{entity_name}/"
    archive_dir = f"{VOLUME_PATH}/{entity_name}/Archive/"
    target_table = f"{TARGET_SCHEMA}.bronze_{entity_name.lower()}"

    # 1. Read top-level files
    raw_df = (
        spark.read
        .option("recursiveFileLookup", "false")
        .format("parquet")
        .load(landing_path)
        .withColumn("bronze_ingestion_time", current_timestamp())
        .withColumn("source_file_name", col("_metadata.file_path"))
    )

    # 2. Process only if files exist
    if not raw_df.isEmpty():
        # Extract distinct processed file paths
        files_to_move = [
            row.source_file_name 
            for row in raw_df.select("source_file_name").distinct().collect()
        ]

        # Append to Bronze Delta Table
        (
            raw_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

        print(f"✅ {entity_name} appended. Total row count: {spark.table(target_table).count()}")

        # Ensure Archive folder exists and move files
        dbutils.fs.mkdirs(archive_dir)
        for file_path in files_to_move:
            file_name = file_path.split("/")[-1]
            dbutils.fs.mv(file_path, f"{archive_dir}{file_name}")

        print(f"📦 Archived {len(files_to_move)} file(s) to {archive_dir}")

    else:
        print(f"ℹ️ No new {entity_name} files found to process.")

In [0]:
# COMMAND ----------
# STEP 2: INGEST CUSTOMERS VIA BATCH LOAD & ARCHIVE PROCESSED FILES
# COMMAND ----------
ingest_and_archive_entity("Customers")

# COMMAND ----------
# STEP 3: INGEST ORDERS VIA BATCH LOAD & ARCHIVE PROCESSED FILES
# COMMAND ----------
ingest_and_archive_entity("Orders")